# Notebook 03 — Entraînement du Modèle ML avec Spark MLlib

**Phase 1 — Batch Training**

Ce notebook entraîne le modèle de classification de sentiment (Logistic Regression binaire) sur le dataset Sentiment140 via **Spark MLlib**.

Pipeline ML :
```
Texte brut → Tokenizer → StopWordsRemover → HashingTF (2^18) → IDF → Logistic Regression
```

**Lien rapport** : Chapitres 4 — Phase 1 (sections 4.3, 4.4, 4.5)

In [ ]:
import sys
import os
from pathlib import Path

# Détection Docker vs Windows
in_docker = os.path.exists('/.dockerenv')
ROOT_DIR = Path('/workspace') if in_docker else Path().absolute().parent
sys.path.insert(0, str(ROOT_DIR))

from src.train_model import build_full_ml_pipeline, run_training
from src.preprocessing import load_sentiment140, clean_dataframe
from src.utils import get_spark_session, get_logger, parquet_exists
from config.config import *

logger = get_logger('training_notebook')
print('✅ Imports OK')
print(f'   Mode    : {"Docker" if in_docker else "Windows local"}')
print(f'   ROOT_DIR: {ROOT_DIR}')
print(f'   MODEL_PATH: {MODEL_PATH}')


## 1. Architecture du Pipeline ML

Le pipeline encapsule TOUT le traitement NLP + le modèle ML en un seul objet sérialisable.
À l'inférence, seule la colonne `text` (brut) est nécessaire.

In [ ]:
# Affichage de l'architecture du pipeline
pipeline = build_full_ml_pipeline()
print('=== PIPELINE SPARK ML ===')
for i, stage in enumerate(pipeline.getStages()):
    print(f'  Stage {i+1}: {stage.__class__.__name__}')
    if hasattr(stage, 'getNumFeatures'):
        print(f'           NumFeatures = {stage.getNumFeatures():,}')
    if hasattr(stage, 'getMaxIter'):
        print(f'           MaxIter = {stage.getMaxIter()}, RegParam = {stage.getRegParam()}')

## 2. Entraînement complet

> ⏱️ **Durée estimée :** 5-15 minutes sur 1,6M tweets selon les ressources disponibles.  
> Pour un test rapide, passez `sample_fraction=0.1` (160K tweets, ~1-2 minutes)

In [ ]:
# Entraînement complet
# Décommentez la ligne 'sample_fraction' pour un test rapide

pipeline_model, metrics = run_training(
    # sample_fraction=0.1  # Décommenter pour test rapide (10% = 160K tweets)
)

print('\n=== RÉSULTATS ===')
for name, val in metrics.items():
    print(f'  {name:<15} : {val:.4f}')

print(f'\n✅ Modèle sauvegardé dans : {MODEL_PATH}')

## 3. Vérification du modèle sauvegardé

In [ ]:
# Structure du modèle sauvegardé
import os

print('=== STRUCTURE DU MODÈLE SAUVEGARDÉ ===')
for root, dirs, files in os.walk(MODEL_PATH):
    level = root.replace(str(MODEL_PATH), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{Path(root).name}/')
    if level < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

## 4. Test d'inférence manuelle

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F
from src.utils import get_spark_session, get_neutral_udf

# get_spark_session utilise SPARK_MASTER depuis config.py
# Docker  : spark://spark-master:7077
# Windows : local[*]
spark = get_spark_session('Inference_Test')
print(f'Master : {spark.sparkContext.master}')

loaded_model = PipelineModel.load(str(MODEL_PATH))
print('✅ Modèle chargé')

# Tweets de test
test_tweets = [
    'I love this sunny day so much!',
    'This is absolutely terrible, worst experience ever.',
    'Just got home from work today.',
    'Feeling a bit tired this morning.',
    'The new iPhone battery life is amazing!',
    'Apple support is the worst I have ever dealt with.'
]

test_df = spark.createDataFrame([(t,) for t in test_tweets], ['text'])
predictions = loaded_model.transform(test_df)

# Application de la couche Neutral
neutral_udf = get_neutral_udf(CONFIDENCE_THRESHOLD)
results = predictions.withColumn(
    'sentiment_label',
    neutral_udf(
        F.col('prediction'),
        F.col('probability').getItem(0),
        F.col('probability').getItem(1)
    )
).withColumn('confidence', F.greatest(
    F.col('probability').getItem(0), F.col('probability').getItem(1)
))

print('=== PRÉDICTIONS SUR TWEETS DE TEST ===')
results.select(
    'text',
    F.col('probability').getItem(0).alias('P(Neg)'),
    F.col('probability').getItem(1).alias('P(Pos)'),
    'confidence',
    'sentiment_label'
).show(truncate=50)

spark.stop()
print('✅ SparkSession fermée')


## 5. Synthèse Phase 1

- ✅ Modèle Logistic Regression entraîné sur 1,28M tweets
- ✅ Pipeline NLP complet intégré (TF-IDF 2^18 features)
- ✅ Modèle sérialisé en Parquet MLlib
- ✅ Couche métier Neutral testée

**Prochaine étape :** `04_model_evaluation_sentiment140.ipynb` pour l'évaluation approfondie.